[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alanturin-g/Computational_Neuroscience_UNITO_2026/blob/main/Orsenigo_2_brain_modeling_applications_solutions.ipynb)

# Lesson 2 — Application of brain modeling: a literature review (solutions)

**Computational Neuroscience 2026/27 · Davide Orsenigo**

**Thursday 5 November 2026, room B**

This notebook accompanies the literature review of Lesson 2. It takes one step beyond Lesson 1, following the
book: Chapter 3 (*generative effective connectivity*, eq. 3.20–3.21) and Chapter 8 (*Thermodynamics of mind*).

> G. Deco and M. L. Kringelbach, *Whole-brain modelling: Cartography of the dynamics of mind*, Oxford University Press (2026), open access (CC BY-NC-ND 4.0), doi:[10.1093/oso/9780198991250.001.0001](https://doi.org/10.1093/oso/9780198991250.001.0001).

The question is how far whole-brain models can go beyond one global coupling. Are resting-state fluctuations the
same when time runs backwards? Can a model recover the direction of the interactions, which diffusion MRI cannot
see? And how much data does that take?

| Part | Book | Content |
|---|---|---|
| 1 | ch. 8, Fig. 8.3 | Time-lagged correlations and the arrow of time |
| 2 | eq. 3.20 | Generative effective connectivity fitted to the empirical FC |
| 3 | eq. 3.20, ch. 10 | Recovering a hidden hierarchy, and how much data it needs |

The notebook runs on the default Colab runtime in about a minute.

Solutions to the two exercises in `Orsenigo_2_brain_modeling_applications`, with answers to the questions.

---
# Setup

Same data as in Lesson 1. The functions of Lesson 1 are collected in the next cell, with one change: the linear
theory is written with complex numbers, $z = x + iy$. It gives the same FC as the $2N \times 2N$ real version of
Lesson 1, but uses $N \times N$ matrices and is faster. From now on $C$ includes $G$, as in the effective
connectivity of the book (p. 46).

In [ ]:
import hashlib
import os
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm, solve_continuous_lyapunov

# Figure style: minimal axes, Arial-like font, palette and colormaps of figkit (D. Orsenigo)
from matplotlib.colors import LinearSegmentedColormap
PALETTE = ["#3B6E8F", "#E07A5F", "#81B29A", "#E0A458", "#8D6A9F", "#C65B6B", "#5B7080", "#3D3D3D"]
FC_CMAP = LinearSegmentedColormap.from_list(   # diverging petrol-brick, white at 0
    "fk_div", ["#2C5F86", "#6FA0BE", "#BFD3DF", "#F3F3F1", "#EFC0B2", "#D77C6A", "#B23A48"])
SC_CMAP = LinearSegmentedColormap.from_list(   # sequential cream-plum
    "fk_warm", ["#FBEEE6", "#F6C19A", "#EE8A5C", "#D8503F", "#A4255B", "#5E1246"])
plt.rcParams.update({
    "figure.dpi": 110, "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "font.size": 10, "axes.titlesize": 10.5, "axes.labelsize": 10, "legend.fontsize": 9,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "mathtext.fontset": "dejavusans",
    "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.8, "axes.grid": False,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.prop_cycle": plt.cycler(color=PALETTE),
    "xtick.direction": "out", "ytick.direction": "out", "xtick.major.size": 3, "ytick.major.size": 3,
    "xtick.major.width": 0.8, "ytick.major.width": 0.8, "lines.linewidth": 1.4, "lines.markersize": 4.5,
    "legend.frameon": False, "image.cmap": "magma"})

ENIGMA = ("https://raw.githubusercontent.com/MICA-MNI/ENIGMA/cf6917af2afcc642d70402fc96ba8c30c711b1a5/"
          "enigmatoolbox/datasets/matrices/hcp_connectivity/")     # one commit of the repository
SHA256 = {"strucMatrix_ctx.csv": "e57b94ec8de0b30f128899413b02a09538a2911633db41fff00ff4715d5841b5",
          "funcMatrix_ctx.csv": "8c4fbbe79dc851ac51bff40bf1bdfd07ce839f819d74519ad7518a731e07b934",
          "strucLabels_ctx.csv": "47a5301cb4f956782fa99c6c0b110fc4717c4bb626a65a4d04890ed72a745a35"}
for file_name, digest in SHA256.items():
    if not os.path.exists(file_name):
        urllib.request.urlretrieve(ENIGMA + file_name, file_name)
    if hashlib.sha256(open(file_name, "rb").read()).hexdigest() != digest:
        print(f"Warning: {file_name} is not the file this notebook was written for")

SC_log = np.loadtxt("strucMatrix_ctx.csv", delimiter=",")
SC = np.where(SC_log > 0, np.exp(SC_log), 0.0)
C = 0.2 * SC / SC.max()
FC_emp = np.tanh(np.loadtxt("funcMatrix_ctx.csv", delimiter=","))
np.fill_diagonal(FC_emp, 1.0)
labels = np.loadtxt("strucLabels_ctx.csv", dtype=str, delimiter=",")
N = len(C)
G = 8.0                                       # the coupling where the mean FC matches the data (Lesson 1)


def upper(M):
    return M[np.triu_indices(len(M), 1)]


def fc_fit(FC_model, FC_data):
    return np.corrcoef(upper(FC_model), upper(FC_data))[0, 1]


def linear_stats(Ceff, tau=0.0, a=-0.02, f=0.05, sigma=0.02):
    """FC and FS(tau) of the linearised Hopf model with connectivity Ceff (G included), eq. 3.19-3.21.
    FS[i, j] = corr(x_i(t + tau), x_j(t)). Complex form: M = diag(a - S + i omega) + Ceff."""
    n = len(Ceff)
    M = np.diag(a - Ceff.sum(axis=1) + 2j * np.pi * np.broadcast_to(f, (n,))) + Ceff
    Sz = solve_continuous_lyapunov(M, -2 * sigma**2 * np.eye(n))    # <z z^H>
    K = Sz.real / 2                                                 # <x x^T>, eq. 3.19
    d = np.sqrt(np.diag(K))
    FC = K / np.outer(d, d)
    if tau == 0:
        return FC, FC
    KS = (expm(tau * M) @ Sz).real / 2                              # <x(t + tau) x(t)^T>, eq. 3.21
    return FC, KS / np.outer(d, d)


def lagged_fc(x, lag=1):
    """Empirical FS from one run x of shape (T, N): corr(x_i(t + lag), x_j(t)), lag in samples.
    Each region is z-scored over the whole run, as for a stationary signal; np.corrcoef of the two shifted
    windows would estimate mean and standard deviation again in each window, with slightly different values."""
    z = (x - x.mean(axis=0)) / x.std(axis=0)
    return z[lag:].T @ z[:-lag] / (len(z) - lag)

In [ ]:
def simulate_hopf(C, G, a=-0.02, f=0.05, sigma=0.02, dt=0.1, TR=2.0, n_vol=600, burn=200.0, seed=0,
                  return_z=False, cubic=True):
    """Whole-brain Hopf model (eq. 3.13-3.14) with Euler-Maruyama; the rotation by omega*dt is applied exactly.
    C: (N, N) fibre strengths; G: one value or an array of values (simulated together); f: frequency in Hz;
    sigma: size of the noise; TR: time between two samples in s (one fMRI image every 2 s); n_vol: number of samples;
    burn: seconds simulated and discarded first, so that the start does not matter.
    cubic=False drops the cubic term -(x^2 + y^2): the linearised model.
    Returns x with shape (len(G), n_vol, N), or z = x + iy if return_z is True."""
    C = np.asarray(C, dtype=float)
    N = C.shape[0]
    G = np.atleast_1d(np.asarray(G, dtype=float))[:, None]          # (B, 1): one row per value of G
    a = np.asarray(a, dtype=float)                                  # a number, (N,) or (B, 1)
    B = max(len(G), a.shape[0] if a.ndim == 2 else 1)               # number of simulations run together
    s = C.sum(axis=1)                                               # strength of each region: sum of its fibres
    w = 2 * np.pi * f
    cw, sw = np.cos(w * dt), np.sin(w * dt)
    rng = np.random.default_rng(seed)
    x, y = 0.1 * rng.standard_normal((2, B, N))
    step, n_burn = int(round(TR / dt)), int(round(burn / dt))
    out = np.empty((B, n_vol, N), dtype=complex if return_z else float)
    for k in range(n_burn + n_vol * step):
        rho2 = x**2 + y**2 if cubic else 0.0                        # squared amplitude of each node
        xe = x + dt * ((a - rho2) * x + G * (x @ C.T - s * x))      # sum_i C_ji (x_i - x_j)
        ye = y + dt * ((a - rho2) * y + G * (y @ C.T - s * y))
        noise = sigma * np.sqrt(dt) * rng.standard_normal((2, B, N))
        x, y = cw * xe - sw * ye + noise[0], sw * xe + cw * ye + noise[1]
        if k >= n_burn and (k - n_burn) % step == 0:
            out[:, (k - n_burn) // step] = x + 1j * y if return_z else x
    return out

In [ ]:
print(f"N = {N} regions; linear FC at G = {G:.0f}: fit to the data = {fc_fit(linear_stats(G * C)[0], FC_emp):.3f}")

---
# Part 1 — Time-lagged correlations and the arrow of time

Consider the time-shifted correlation $FS_{ij}(\tau) = \mathrm{corr}\big(x_i(t + \tau),\, x_j(t)\big)$. If we play the
recording backwards, $FS(\tau)$ becomes its transpose: the past of $i$ and the future of $j$ swap roles. A
stationary process in **detailed balance** (time-reversible, at equilibrium) looks the same backwards, so its
$FS(\tau)$ must be symmetric at every $\tau$. The converse does not hold in general: a symmetric $FS$ at one lag
does not prove that the whole process is reversible.

The book measures how far a brain state is from equilibrium by comparing the forward and the reversed series
(INSIDEOUT, Fig. 8.3D, drawn as the norm of the difference of their shifted correlations). We use the simplest
version of this idea, a teaching index of the asymmetry of the shifted correlations:

$$I(\tau) = \big\lVert FS_{fwd}(\tau) - FS_{rev}(\tau) \big\rVert = \big\lVert FS(\tau) - FS(\tau)^T \big\rVert .$$

The original INSIDEOUT (Deco et al. 2022) first turns each shifted correlation $c$ into the mutual information of
a Gaussian pair, $-\tfrac12 \ln(1 - c^2)$ (Lesson 0), and then averages the squared differences over all pairs. Its
values are on another scale, and a difference of sign alone does not count. The cell after Exercise 1 prints both.

In the linearised Hopf model the symmetry can be broken by **asymmetric connections** ($C_{ij} \neq C_{ji}$) or by
**different intrinsic frequencies** $\omega_j$ (different noise levels would also do it; we keep them equal). The
next cell builds three cases.

In [ ]:
rng = np.random.default_rng(0)
h_true = rng.standard_normal(N)               # a hidden "hierarchy": one number per region
C_sym = G * C
C_asym = G * C * np.exp(0.4 * (h_true[:, None] - h_true[None, :]))   # j -> i stronger if h_i > h_j
f_het = rng.uniform(0.04, 0.07, N)            # intrinsic frequencies between 0.04 and 0.07 Hz

def insideout(FS):
    """Irreversibility as in Deco et al. (2022): the shifted correlations turned into Gaussian MI (nats),
    then the mean squared asymmetry over all pairs."""
    M = -0.5 * np.log(1 - FS**2)
    return np.mean((M - M.T) ** 2)


cases = {"symmetric C, equal frequencies": (C_sym, 0.05),
         "symmetric C, different frequencies": (C_sym, f_het),
         "asymmetric C, equal frequencies": (C_asym, 0.05)}
tau = 2.0                                     # one TR

## Exercise 1: Measuring irreversibility

Write `irreversibility(FS)`, the norm of $FS - FS^T$ (`np.linalg.norm` of a matrix is its Frobenius norm). Before
running the cell after it, predict which of the three cases gives zero.

*Expected:* zero (to machine precision) for the first case, about 3.5 for the second, about 0.8 for the third.
These are values of our index. The original INSIDEOUT, printed next to it, gives 0, 1.2e-4 and 3.7e-5: the same
order of the three cases, on another scale.

**Solution**

In [ ]:
def irreversibility(FS):
    return np.linalg.norm(FS - FS.T)

In [ ]:
for name, (Ceff, f) in cases.items():
    FC, FS = linear_stats(Ceff, tau, f=f)
    print(f"{name:36s} I = {irreversibility(FS):.3f}   INSIDEOUT = {insideout(FS):.1e}")

**Question 1.** In the first case each region still oscillates in its $(x, y)$ plane, always in the same
direction: the full system is *not* in equilibrium, and its entropy production is positive. Yet the BOLD-like
signal $x$ looks perfectly reversible. Why? And what does the second case tell you about interpreting
irreversibility as "directed interactions"?

**Answer.** We only observe $x$, a projection of the state $(x, y)$. With symmetric $C$ and equal frequencies the rotation is
the same everywhere and commutes with the rest of the dynamics, so it leaves no trace in the correlations of $x$.
The asymmetry of the shifted correlations detects the part of the irreversibility that is visible in $x$; it does
not measure all of it, and here it is not converted into entropy production. (The results that bound the entropy
production from partial observations, as in Lynn et al. 2021, concern the divergence between the probabilities
of forward and backward trajectories, not this norm.) In the second case the
connections are symmetric, yet $x$ is irreversible: regions with different intrinsic frequencies shift their
phases relative to each other. Measured irreversibility therefore does not identify asymmetric connections by
itself. A model is needed to tell the two causes apart, which is why the book fits the GEC with the frequencies
estimated from the data.

The numbers above are exact for the linearised model: the linear theory corresponds to infinitely long
recordings of it. The next cell estimates
the same quantity from simulations of 20 minutes (600 volumes, TR = 2 s), the length of a typical resting-state
scan, in the reversible case where the true value is zero.

In [ ]:
X = simulate_hopf(C, np.full(20, G), seed=1)  # 20 independent runs of the reversible case
single = [irreversibility(lagged_fc(x)) for x in X]
averaged = irreversibility(np.mean([lagged_fc(x) for x in X], axis=0))
print(f"one run of 20 min:      I = {np.mean(single):.2f} +- {np.std(single):.2f}   (true value 0)")
print(f"FS averaged over 20 runs: I = {averaged:.2f}")

**Question 2.** The estimate from a single scan is larger than the exact irreversibility of the asymmetric model.
What does this imply for studies that compare the "arrow of time" between groups or conditions? What would you
compare the measured value with?

**Answer.** Finite-sample noise makes $FS(\tau)$ asymmetric even for a reversible process, and with one scan this noise is
larger than the effect. A measured value means something only against a null distribution with the same amount
of data and the same autocorrelation, which sets the size of this noise. One option is a reversible model fitted
to the same data and simulated with the same duration and TR, as in the cell above, once it is checked that it
reproduces the FC and the autocorrelation of the data. Surrogates that shuffle the time points are not a
calibrated null here, because they destroy the autocorrelation too.
It also needs averaging over many subjects. The book's results use the HCP data, with about a thousand
participants.

---
# Part 2 — Generative effective connectivity (eq. 3.20)

In Lesson 1 one parameter, $G$, scaled all connections. The **generative effective connectivity** (GEC) gives each
existing connection its own weight and fits all of them with a pseudo-gradient descent (p. 46):

$$C_{ij} \leftarrow C_{ij} + \alpha\big(FC^{emp}_{ij} - FC^{model}_{ij}\big) + \varsigma\big(FS^{emp}_{ij}(\tau) - FS^{model}_{ij}(\tau)\big). \qquad (3.20)$$

Only connections present in the SC are updated, plus the homotopic ones (the same region in the two hemispheres),
which tractography tends to miss. Here we have only the empirical FC, so we use the first term ($\varsigma = 0$)
and the GEC stays symmetric. The book quotes $\alpha = \varsigma = 10^{-5}$ for its own normalisation; with our $C$
a step of 0.02 converges in a few hundred iterations.

In [ ]:
homotopic = np.array([list(labels).index(("R_" if l.startswith("L_") else "L_") + l[2:]) for l in labels])
mask = C > 0                                  # connections that the GEC may change
mask[np.arange(N), homotopic] = True
np.fill_diagonal(mask, False)
print(f"{upper(C > 0).sum()} connections in the SC, {upper(mask).sum()} in the mask")

## Exercise 2: Fitting the GEC

Complete the loop with eq. 3.20 (first term only): update `Ceff` on the connections of `mask`, and keep the
weights non-negative with `np.clip(..., 0, None)`.

*Expected:* the fit grows from 0.66 to about 0.91 in 300 iterations (a few seconds).

**Solution**

In [ ]:
alpha = 0.02
Ceff = G * C.copy()                           # start from the SC scaled by G
history = []
for iteration in range(300):
    FC_model, _ = linear_stats(Ceff)
    history.append(fc_fit(FC_model, FC_emp))
    Ceff = np.clip(Ceff + alpha * (FC_emp - FC_model) * mask, 0, None)

In [ ]:
exists = C > 0
pruned = ((Ceff == 0) & exists).sum() / exists.sum()
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(history)
axes[0].set_xlabel("iteration")
axes[0].set_ylabel("fit")
axes[1].loglog(G * C[exists], Ceff[exists] + 1e-4, ".", ms=3, alpha=0.5, label="SC connections")
axes[1].loglog(G * C[np.arange(N), homotopic] + 1e-4, Ceff[np.arange(N), homotopic] + 1e-4, "o",
               ms=4, label="homotopic")
axes[1].set_xlabel("G x SC")
axes[1].set_ylabel("GEC (+ 1e-4)")
axes[1].legend(loc="lower right")
plt.tight_layout()
plt.show()
print(f"final fit {history[-1]:.3f}; {100 * pruned:.0f}% of the SC connections were set to zero")

**Question 3.** The GEC fits the FC far better, but it has one free parameter per connection (about 700) for about
2300 FC values, and it switched off half of the anatomical connections. How would you check that the fitted
weights mean something, rather than just absorbing noise?

**Answer.** Fit the GEC on one half of the subjects and test it on the other half (cross-validation), or check that it is
stable across sessions and initialisations. Compare it, on the held-out data, with simpler models: one global $G$,
or a GEC with fewer free connections. Predict something it was not fitted to: lagged correlations, the response
to stimulation or task, the effect of a drug. Surrogate data with the same FC would not help here: the fit sees
only the FC, so it would solve the same problem and return the same GEC. Many weak tractography connections go to zero because the update is additive: a
small weight cannot survive a step of 0.02 times an FC error of about 0.1. This says more about the algorithm and
its normalisation than about the anatomy.

---
# Part 3 — Recovering a hidden hierarchy

With the second term of eq. 3.20, the time-lagged correlations, the GEC becomes asymmetric. The book uses it to
measure the hierarchy of brain regions (ch. 4, 8 and 10): regions that mostly send are at the bottom, regions that
mostly receive at the top. The book makes this precise with **trophic levels** (ch. 10, eq. 10.3–10.8; MacKay et
al. 2020). With $C_{nm}$ the weight from $m$ to $n$, $d^{in}_n = \sum_m C_{nm}$ and $d^{out}_n = \sum_m C_{mn}$, the
levels $h$ solve

$$\big[\mathrm{diag}\big(d^{in} + d^{out}\big) - C - C^T\big]\, h = d^{in} - d^{out}.$$

A region with a high level receives more than it sends. (In ch. 10 the second term of eq. 3.20 is replaced by its
antisymmetric part, $FS - FS^T$, eq. 10.1: the irreversibility of Part 1.)

We test the method on data generated by the asymmetric model of Part 1, whose hierarchy we know. The GEC is
fitted three times: to the exact FC and FS of the linearised model (infinite data), and to estimates from 10 and
from 100 runs of 20 minutes. The runs are simulated without the cubic term (`cubic=False`), so that the three
targets differ only in the amount of data (and, slightly, in the time step of the simulation).

In [ ]:
def trophic_levels(Ceff):
    """Trophic levels h (eq. 10.3-10.8). Ceff[n, m] is the weight from m to n, as in the book."""
    d_in, d_out = Ceff.sum(axis=1), Ceff.sum(axis=0)
    Lam = np.diag(d_in + d_out) - Ceff - Ceff.T
    h = np.linalg.lstsq(Lam, d_in - d_out, rcond=None)[0]          # defined up to a constant
    return h - h.mean()


def fit_gec(FC_target, FS_target, tau=2.0, n_iter=500, step=0.04):
    """GEC with both terms of eq. 3.20 (alpha = varsigma = step), starting from G x SC."""
    Ceff = G * C.copy()
    for _ in range(n_iter):
        FC_model, FS_model = linear_stats(Ceff, tau)
        Ceff = np.clip(Ceff + step * ((FC_target - FC_model) + (FS_target - FS_model)) * mask, 0, None)
    return Ceff


def recovery(Ceff):
    """Correlation with the truth of the asymmetric part (C - C^T) and of the trophic levels."""
    asym = np.corrcoef(upper(Ceff - Ceff.T), upper(C_asym - C_asym.T))[0, 1]
    levels = np.corrcoef(trophic_levels(Ceff), trophic_levels(C_asym))[0, 1]
    return asym, levels


targets = {"exact (infinite data)": linear_stats(C_asym, tau)}
for n_runs in (10, 100):
    X = simulate_hopf(C_asym / G, np.full(n_runs, G), seed=2, cubic=False)   # n_runs scans of 20 min
    targets[f"{n_runs} runs of 20 min"] = (np.mean([np.corrcoef(x.T) for x in X], axis=0),
                                            np.mean([lagged_fc(x) for x in X], axis=0))

for name, (FC_t, FS_t) in targets.items():
    asym, levels = recovery(fit_gec(FC_t, FS_t))
    print(f"{name:24s} asymmetry r = {asym:.2f},  trophic levels r = {levels:.2f}")

**Question 4.** How does the amount of data limit what the asymmetric GEC can recover? What does this suggest about
using it for single patients, as in the clinical studies of ch. 10?

**Answer.** With exact correlations the hierarchy is recovered well. With ten scans the time-lagged correlations are dominated
by sampling noise (Part 1) and the recovered hierarchy is only partly right; with a hundred it improves. Group
studies with hundreds of subjects can estimate a hierarchy. Single-subject or small clinical samples need longer
recordings, priors from group data, and an explicit estimate of the uncertainty before any claim is made. With the
cubic term (`cubic=True`) the recovery from 100 runs is similar, 0.80 and 0.83 instead of 0.81 and 0.87: here the
amount of data matters more than the nonlinearity.

---
# Summary and reading

- Time-lagged correlations, unlike the FC, can be asymmetric. Their asymmetry detects irreversibility in what we
  observe, without measuring all of it. Asymmetric connections and different intrinsic frequencies both
  produce it.
- With one scan the estimate is dominated by noise: irreversibility measures need a lot of data, and a null
  model with the same amount of data and the same autocorrelation.
- The GEC gives every connection its own weight. Fitted to FC it is symmetric and fits much better than one $G$;
  fitted with the lagged correlations it becomes asymmetric and can reveal a hierarchy, given enough data.

**Reading for the review** (book chapters and the original papers):
- ch. 2: model-free measures used to fit models (FCD, metastability, LEiDA);
- ch. 4: hierarchy and the global workspace (Deco, Vidaurre and Kringelbach 2021, *Nat Hum Behav*);
- ch. 5: neuromodulation, LSD and serotonin receptor maps (Deco et al. 2018, *Curr Biol*);
- ch. 6: regional heterogeneity from gene expression (Deco et al. 2021, *Sci Adv*);
- ch. 7: forcing transitions between brain states (Deco et al. 2019, *PNAS*);
- ch. 8: thermodynamics and the arrow of time (Lynn et al. 2021, *PNAS*; Deco et al. 2022, *Commun Biol*);
- ch. 9: turbulence (Deco and Kringelbach 2020, *Cell Rep*);
- ch. 10: hierarchy in depression, psilocybin and escitalopram (Deco et al. 2024, *Nat Ment Health*).

**References for this notebook**
- Gilson M, Moreno-Bote R, Ponce-Alvarez A, Ritter P, Deco G (2016). Estimation of directed effective connectivity from fMRI functional connectivity hints at asymmetries of cortical connectome. *PLoS Comput Biol* 12:e1004762.
- Lynn CW, Cornblath EJ, Papadopoulos L, Bertolero MA, Bassett DS (2021). Broken detailed balance and entropy production in the human brain. *PNAS* 118:e2109889118.
- Deco G, Sanz Perl Y, Bocaccio H, Tagliazucchi E, Kringelbach ML (2022). The INSIDEOUT framework provides precise signatures of the balance of intrinsic and extrinsic dynamics in brain states. *Commun Biol* 5:572.
- MacKay RS, Johnson S, Sansom B (2020). How directed is a directed network? *R Soc Open Sci* 7:201138.
- Nartallo-Kaluarachchi R, Kringelbach ML, Deco G, Lambiotte R, Goriely A (2026). Nonequilibrium physics of brain dynamics. *Phys Rep* 1152:1–43.